# DANA Review Insights — Analisis & Visualisasi (Orang 3)

Notebook ini **siap dijalankan** untuk Orang 3. Input-nya satu file hasil NLP dari Orang 2: `data/processed/reviews_topics_sentiment.csv`
(kalau file lokal tidak ada, otomatis diambil dari GitHub). Tidak perlu melatih model apa pun.

Keluaran: grafik di `reports/figures/` dan tabel `reports/tabel_insight.csv`, yang bisa langsung dipakai di laporan atau dashboard.

**Kolom penting dari Orang 2**

| Kolom | Arti |
|---|---|
| `topic_label` | topik dominan ulasan (8 topik LDA) |
| `topic_prob`, `low_conf` | keyakinan topik; `low_conf = True` berarti kurang yakin (umumnya ulasan sangat pendek) |
| `sentimen` | sentimen final: **IndoBERT/RoBERTa** (positif/netral/negatif). Pakai kolom ini. |
| `sentimen_tfidf` | baseline pembanding. **Jangan dipakai** untuk laporan utama. |
| `versi_grp` | versi aplikasi yang sudah dikelompokkan |
| `is_spike` | hari lonjakan 6 Nov (±33% data) |

Peringatan penting (baca sebelum menulis kesimpulan): lihat bagian **Catatan keterbatasan** di akhir notebook.

In [ ]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

LOCAL = 'data/processed/reviews_topics_sentiment.csv'
URL   = 'https://raw.githubusercontent.com/adheliaissabel/dana-review-insights/main/data/processed/reviews_topics_sentiment.csv'
FIG_DIR = 'reports/figures'; os.makedirs(FIG_DIR, exist_ok=True)

EXCLUDE_LOW_CONF = False   # True = buang ulasan yang topiknya kurang yakin dari semua grafik
MIN_N = 50                 # ambang jumlah ulasan minimum agar persentase per kelompok dianggap cukup andal

D = pd.read_csv(LOCAL if os.path.exists(LOCAL) else URL)
D['date'] = pd.to_datetime(D['date'])
D['tgl'] = D.date.dt.normalize()
if EXCLUDE_LOW_CONF: D = D[~D.low_conf]
ORDER = ['negatif', 'netral', 'positif']
COL = {'negatif': '#d1495b', 'netral': '#b8b8b8', 'positif': '#2a9d8f'}
print(D.shape, '| rentang:', D.date.min().date(), '-', D.date.max().date())
print('Sentimen:', D.sentimen.value_counts(normalize=True).round(3).to_dict())
D[['review_id','topic_label','sentimen','versi_grp','is_spike','low_conf']].head()

## 1. Gambaran umum: porsi topik

Dibandingkan dua kondisi: semua data dan tanpa hari lonjakan 6 Nov. Selisihnya menunjukkan topik mana yang "terdongkrak" oleh satu kejadian.

In [ ]:
a = D.topic_label.value_counts(normalize=True)
b = D[~D.is_spike].topic_label.value_counts(normalize=True)
T = pd.DataFrame({'semua_data': a, 'tanpa_lonjakan_6Nov': b}).fillna(0).sort_values('semua_data')
ax = T.plot.barh(figsize=(8, 5), color=['#264653', '#e9c46a'])
ax.set_xlabel('Porsi ulasan'); ax.set_ylabel(''); ax.set_title('Porsi topik ulasan DANA')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/01_porsi_topik.png'); plt.show()
(T * 100).round(1).sort_values('semua_data', ascending=False)

## 2. Sentimen per topik

In [ ]:
S = pd.crosstab(D.topic_label, D.sentimen, normalize='index')[ORDER]
S['n'] = D.groupby('topic_label').size()
S = S.sort_values('negatif')
ax = S[ORDER].plot.barh(stacked=True, figsize=(9, 5), color=[COL[c] for c in ORDER])
for i, (idx, r) in enumerate(S.iterrows()):
    ax.text(1.01, i, f"n={int(r.n):,}", va='center', fontsize=8)
ax.set_xlim(0, 1.12); ax.set_ylabel(''); ax.set_xlabel('Porsi sentimen'); ax.set_title('Sentimen per topik (IndoBERT/RoBERTa)')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}' if v <= 1 else ''))
ax.legend(title='', ncol=3, loc='upper center', bbox_to_anchor=(0.45, -0.12), frameon=False); plt.tight_layout(); plt.savefig(f'{FIG_DIR}/02_sentimen_per_topik.png', bbox_inches='tight'); plt.show()
S.drop(columns='n').round(3).assign(n=S.n).sort_values('negatif', ascending=False)

## 3. Tren harian per topik

Garis putus-putus menandai lonjakan 6 Nov. Gunakan grafik ini untuk menunjukkan bahwa lonjakan didominasi topik gangguan jaringan.

In [ ]:
daily = pd.crosstab(D.tgl, D.topic_label)
ax = daily.plot.area(figsize=(11, 5), linewidth=0, colormap='tab20')
spike_day = D[D.is_spike].tgl.mode()[0]
ax.axvline(spike_day, ls='--', c='k', lw=1); ax.text(spike_day, ax.get_ylim()[1]*0.97, f' lonjakan {spike_day:%d %b}', va='top', fontsize=9)
ax.set_ylabel('Jumlah ulasan'); ax.set_xlabel(''); ax.set_title('Jumlah ulasan per hari menurut topik')
ax.legend(fontsize=7, ncol=2, loc='upper left', bbox_to_anchor=(1.01, 1)); plt.tight_layout()
plt.savefig(f'{FIG_DIR}/03_tren_harian_topik.png', bbox_inches='tight'); plt.show()
print('Hari lonjakan:', spike_day.date(), '| jumlah ulasan:', int(daily.loc[spike_day].sum()))

In [ ]:
# Tren tanpa hari lonjakan (lebih jelas untuk melihat pola hari biasa)
d2 = daily.drop(index=spike_day)
ax = d2.plot(figsize=(11, 4.5), colormap='tab20', lw=1.5)
ax.set_ylabel('Jumlah ulasan'); ax.set_xlabel(''); ax.set_title('Tren harian per topik (tanpa 6 Nov)')
ax.legend(fontsize=7, ncol=2, loc='upper left', bbox_to_anchor=(1.01, 1)); plt.tight_layout()
plt.savefig(f'{FIG_DIR}/04_tren_tanpa_lonjakan.png', bbox_inches='tight'); plt.show()

## 4. Per versi aplikasi (indikasi bug terbaru)

**Penting:** versi 2.151.0 juga versi yang aktif saat lonjakan 6 Nov, jadi **selalu bandingkan tanpa hari lonjakan** supaya kesimpulan tentang versi tidak bercampur dengan efek lonjakan.
Kelompok dengan ulasan kurang dari `MIN_N` ditandai dan sebaiknya tidak dijadikan kesimpulan.

In [ ]:
def ver_table(df):
    t = df.groupby('versi_grp').agg(n=('review_id','size'), rating=('rating','mean'),
                                    pct_negatif=('sentimen', lambda s: (s == 'negatif').mean()))
    t['andal'] = t.n >= MIN_N
    return t.round(3).sort_values('n', ascending=False)
vt_all, vt_ns = ver_table(D), ver_table(D[~D.is_spike])
print('=== Semua data ==='); display(vt_all)
print('=== Tanpa hari lonjakan ==='); display(vt_ns)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
idx = [v for v in vt_ns.index if vt_ns.loc[v, 'andal']]
x = np.arange(len(idx)); w = 0.38
ax.bar(x - w/2, vt_all.reindex(idx).pct_negatif, w, label='Semua data', color='#264653')
ax.bar(x + w/2, vt_ns.reindex(idx).pct_negatif, w, label='Tanpa 6 Nov', color='#e9c46a')
ax.set_xticks(x); ax.set_xticklabels(idx, rotation=20); ax.set_ylabel('Porsi sentimen negatif')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}')); ax.legend(); ax.set_title('Sentimen negatif per versi aplikasi')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/05_negatif_per_versi.png'); plt.show()

In [ ]:
# Topik apa yang menonjol di tiap versi? (tanpa hari lonjakan; hanya versi yang cukup besar)
N = D[~D.is_spike]
ok = [v for v in N.versi_grp.value_counts()[lambda s: s >= MIN_N].index if v != 'kosong']
H = pd.crosstab(N[N.versi_grp.isin(ok)].topic_label, N[N.versi_grp.isin(ok)].versi_grp, normalize='columns')
fig, ax = plt.subplots(figsize=(1.6*len(ok)+3, 5))
im = ax.imshow(H.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(H.columns))); ax.set_xticklabels(H.columns, rotation=20)
ax.set_yticks(range(len(H.index))); ax.set_yticklabels(H.index)
for i in range(H.shape[0]):
    for j in range(H.shape[1]): ax.text(j, i, f'{H.values[i,j]:.0%}', ha='center', va='center', fontsize=8)
ax.set_title('Porsi topik per versi (tanpa 6 Nov)'); plt.colorbar(im, fraction=0.03); plt.tight_layout()
plt.savefig(f'{FIG_DIR}/06_topik_per_versi.png'); plt.show()

In [ ]:
# Kandidat "bug terbaru": topik yang porsinya di versi terbaru jauh lebih tinggi daripada versi lain (tanpa lonjakan)
newest = max([v for v in ok if v[0].isdigit()], key=lambda v: tuple(int(p) for p in v.split('.')))
cur, rest = N[N.versi_grp == newest], N[(N.versi_grp != newest) & N.versi_grp.isin(ok)]
cmp = pd.DataFrame({f'{newest}': cur.topic_label.value_counts(normalize=True),
                    'versi_lain': rest.topic_label.value_counts(normalize=True)}).fillna(0)
cmp['selisih_poin'] = ((cmp[newest] - cmp.versi_lain) * 100).round(1)
print(f'Versi terbaru: {newest} (n={len(cur)}) vs versi lain (n={len(rest)}), tanpa hari lonjakan')
cmp.sort_values('selisih_poin', ascending=False).round(3)

## 5. Kutipan ulasan per topik

Kutipan diambil dari ulasan dengan **thumbs up terbanyak** dan topik yang yakin (bukan `low_conf`). Cocok untuk ditaruh di laporan atau dashboard.

In [ ]:
Q = (D[~D.low_conf].sort_values(['thumbs_up', 'topic_prob'], ascending=False)
       .groupby('topic_label').head(3)[['topic_label', 'sentimen', 'thumbs_up', 'text_raw']])
Q['text_raw'] = Q.text_raw.str.slice(0, 220)
pd.set_option('display.max_colwidth', 220)
Q.sort_values(['topic_label', 'thumbs_up'], ascending=[True, False]).reset_index(drop=True)

## 6. Tabel insight ringkas (untuk laporan)

In [ ]:
tot = len(D)
I = D.groupby('topic_label').agg(jumlah=('review_id', 'size'), rating_rata2=('rating', 'mean'),
                                  pct_negatif=('sentimen', lambda s: (s == 'negatif').mean()),
                                  pct_hari_lonjakan=('is_spike', 'mean'), pct_kurang_yakin=('low_conf', 'mean'))
I['porsi'] = I.jumlah / tot
I = I[['jumlah', 'porsi', 'rating_rata2', 'pct_negatif', 'pct_hari_lonjakan', 'pct_kurang_yakin']].round(3).sort_values('jumlah', ascending=False)
I.to_csv('reports/tabel_insight.csv'); I

## Catatan keterbatasan (wajib disebut di laporan)

1. **Topik cocok manual hanya ±49%** (±54% untuk ulasan yang yakin) pada sampel 100 ulasan. Topik adalah **perkiraan**, bukan fakta. Yang paling konsisten: *Pujian Umum* (±71% cocok); *Gangguan Jaringan* hanya ±50% cocok karena banyak keluhan jaringan juga menyebut login/transaksi.
   Yang sering tertukar: ulasan "tidak bisa masuk akun" dengan "uang tidak masuk" (kata "masuk" ambigu), sehingga *Login, Akun & Update* dan *Transfer/Top-up Tidak Masuk* kurang tegas.
2. **Sentimen**: akurasi ±72% dan macro-F1 ±0,68 terhadap label manual (n = 100). Negatif dan positif cukup baik, tetapi **netral lemah** (banyak ulasan netral terbaca negatif),
   jadi porsi negatif cenderung **terlalu tinggi**. Selisih kecil antar kelompok (beberapa poin persen) jangan ditafsirkan sebagai perbedaan nyata.
3. **Lonjakan 6 Nov (±33% data)** mendongkrak topik gangguan jaringan dan versi 2.151.0. Selalu tampilkan juga hasil tanpa `is_spike`.
4. **Data hanya ±17 hari**, jadi tren jangka panjang tidak bisa disimpulkan. Perbandingan versi memakai satu versi dominan, sehingga bukti "bug versi terbaru" bersifat **indikasi**, bukan bukti kausal.
5. **±28% ulasan `low_conf`** (sangat pendek). Coba `EXCLUDE_LOW_CONF = True` untuk memeriksa apakah kesimpulan tetap sama.
6. Rating bukan sentimen: ada keluhan yang diberi bintang 5.